# UmarTransit-1B: Benchmark Evaluation

Compare **UmarTransit-1B** (fine-tuned) vs **Qwen2.5-1.5B-Instruct** (base model) on 193 structured benchmark questions.

**Categories:** GTFS Terminology, GTFS Validation, Route Analysis, Journey Planning, Schedule Reasoning, Transit Operations

**Metrics:** ROUGE-L, Keyword Match, Criteria Match, Combined Score

> **Runtime:** Select **T4 GPU** from Runtime → Change runtime type. Total time ~20-30 min.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers torch accelerate rouge-score nltk tqdm

## 2. Download Benchmark & Metrics from GitHub

In [ ]:
import requests
import os

GITHUB_RAW = "https://raw.githubusercontent.com/umarfarookm/transit-foundation-model/main"

files_to_download = {
    "benchmark.json": f"{GITHUB_RAW}/evaluation/benchmark.json",
    "metrics.py": f"{GITHUB_RAW}/evaluation/metrics.py",
}

for filename, url in files_to_download.items():
    print(f"Downloading {filename}...")
    r = requests.get(url)
    if r.status_code == 200:
        with open(filename, "w") as f:
            f.write(r.text)
        print(f"  ✓ Saved ({len(r.text):,} bytes)")
    else:
        print(f"  ✗ Failed (HTTP {r.status_code}) — upload manually from evaluation/ folder")

# Verify
for f in files_to_download:
    print(f"  {f}: {'exists' if os.path.exists(f) else 'MISSING'}")

> **If download failed:** Upload `benchmark.json` and `metrics.py` manually from `evaluation/` folder using the Colab file browser (left panel).

## 3. Configuration

In [ ]:
import json
import torch

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
UMARTRANSIT_MODEL_ID = "umarfarookm/UmarTransit-1B"

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1
TOP_P = 0.9

SYSTEM_PROMPT = (
    "You are UmarTransit-1B, a specialized AI assistant for public transit systems "
    "and GTFS (General Transit Feed Specification) data. You provide accurate, "
    "detailed answers about transit routes, stops, schedules, transfers, and GTFS concepts."
)

BASE_SYSTEM_PROMPT = (
    "You are a helpful AI assistant that answers questions about public transit systems "
    "and GTFS (General Transit Feed Specification) data."
)

# Load benchmark
with open("benchmark.json") as f:
    benchmark = json.load(f)

questions = benchmark["questions"]
print(f"Benchmark: {benchmark['metadata']['name']}")
print(f"Questions: {len(questions)}")
print(f"Categories: {benchmark['metadata']['categories']}")
print(f"Device: {'CUDA (' + torch.cuda.get_device_name() + ')' if torch.cuda.is_available() else 'CPU'}")

## 4. Load Metrics Module

In [ ]:
from metrics import score_response, score_benchmark, print_report, report_to_dict

## 5. Inference Helper

In [ ]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


def load_model(model_id):
    """Load model onto GPU with bfloat16."""
    print(f"Loading {model_id}...")
    start = time.time()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    model.eval()
    elapsed = time.time() - start
    mem = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    print(f"  Loaded in {elapsed:.1f}s | GPU memory: {mem:.1f} GB")
    return model, tokenizer


def generate_response(model, tokenizer, question, system_prompt):
    """Generate a single response."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def run_evaluation(model, tokenizer, questions, model_id, system_prompt):
    """Evaluate all benchmark questions."""
    results = []
    for q in tqdm(questions, desc=model_id.split('/')[-1]):
        start = time.time()
        generated = generate_response(model, tokenizer, q["question"], system_prompt)
        elapsed = time.time() - start

        score = score_response(
            generated=generated,
            expected=q["expected_answer"],
            scoring_criteria=q["scoring_criteria"],
        )
        results.append({
            "id": q["id"],
            "question": q["question"],
            "expected_answer": q["expected_answer"],
            "generated": generated,
            "scoring_criteria": q["scoring_criteria"],
            "category": q["category"],
            "difficulty": q["difficulty"],
            "rouge_l": round(score.rouge_l, 4),
            "keyword_match": round(score.keyword_match, 4),
            "criteria_match": round(score.criteria_match, 4),
            "combined": round(score.combined, 4),
            "time_seconds": round(elapsed, 2),
        })
    return results

## 6. Evaluate Base Model (Qwen2.5-1.5B-Instruct)

~10-15 min on T4 GPU.

In [ ]:
base_model, base_tokenizer = load_model(BASE_MODEL_ID)
base_results = run_evaluation(base_model, base_tokenizer, questions, BASE_MODEL_ID, BASE_SYSTEM_PROMPT)

base_report = score_benchmark(base_results, model_id=BASE_MODEL_ID)
print_report(base_report)
base_report_dict = report_to_dict(base_report)

# Free GPU memory
del base_model, base_tokenizer
torch.cuda.empty_cache()
print("\nBase model unloaded, GPU memory freed.")

## 7. Evaluate UmarTransit-1B

~10-15 min on T4 GPU.

In [ ]:
umar_model, umar_tokenizer = load_model(UMARTRANSIT_MODEL_ID)
umar_results = run_evaluation(umar_model, umar_tokenizer, questions, UMARTRANSIT_MODEL_ID, SYSTEM_PROMPT)

umar_report = score_benchmark(umar_results, model_id=UMARTRANSIT_MODEL_ID)
print_report(umar_report)
umar_report_dict = report_to_dict(umar_report)

# Free GPU memory
del umar_model, umar_tokenizer
torch.cuda.empty_cache()
print("\nUmarTransit model unloaded, GPU memory freed.")

## 8. Side-by-Side Comparison

In [ ]:
print("=" * 80)
print("  SIDE-BY-SIDE COMPARISON: Base Model vs UmarTransit-1B")
print("=" * 80)

print(f"\n  {'Metric':<25} {'Base Model':>15} {'UmarTransit':>15} {'Delta':>10}")
print("  " + "-" * 65)

for metric in ["rouge_l", "keyword_match", "criteria_match", "combined"]:
    base_val = base_report_dict["overall"][metric]
    umar_val = umar_report_dict["overall"][metric]
    delta = umar_val - base_val
    sign = "+" if delta > 0 else ""
    print(f"  {metric:<25} {base_val:>15.4f} {umar_val:>15.4f} {sign}{delta:>9.4f}")

print(f"\n  {'Category':<25} {'Base Combined':>15} {'Umar Combined':>15} {'Delta':>10}")
print("  " + "-" * 65)

all_cats = sorted(set(list(base_report_dict["categories"].keys()) + list(umar_report_dict["categories"].keys())))
for cat in all_cats:
    base_val = base_report_dict["categories"].get(cat, {}).get("combined_avg", 0)
    umar_val = umar_report_dict["categories"].get(cat, {}).get("combined_avg", 0)
    delta = umar_val - base_val
    sign = "+" if delta > 0 else ""
    print(f"  {cat:<25} {base_val:>15.4f} {umar_val:>15.4f} {sign}{delta:>9.4f}")

print("=" * 80)

overall_delta = umar_report_dict["overall"]["combined"] - base_report_dict["overall"]["combined"]
if overall_delta > 0.05:
    verdict = "UmarTransit-1B SIGNIFICANTLY OUTPERFORMS the base model."
elif overall_delta > 0:
    verdict = "UmarTransit-1B SLIGHTLY OUTPERFORMS the base model."
elif overall_delta > -0.05:
    verdict = "Performance is ROUGHLY EQUIVALENT."
else:
    verdict = "The BASE MODEL outperforms UmarTransit-1B."

print(f"\n  VERDICT: {verdict} (delta = {overall_delta:+.4f})")

## 9. Sample Predictions (Best & Worst)

In [ ]:
# Show best and worst UmarTransit predictions per category
from collections import defaultdict

by_cat = defaultdict(list)
for r in umar_results:
    by_cat[r["category"]].append(r)

for cat in sorted(by_cat.keys()):
    cat_items = sorted(by_cat[cat], key=lambda x: x["combined"], reverse=True)
    best = cat_items[0]
    worst = cat_items[-1]

    print(f"\n{'='*70}")
    print(f"  {cat.upper()} — Best (combined={best['combined']:.3f})")
    print(f"{'='*70}")
    print(f"  Q: {best['question'][:120]}")
    print(f"  Expected:  {best['expected_answer'][:150]}")
    print(f"  Generated: {best['generated'][:150]}")

    print(f"\n  {cat.upper()} — Worst (combined={worst['combined']:.3f})")
    print(f"  Q: {worst['question'][:120]}")
    print(f"  Expected:  {worst['expected_answer'][:150]}")
    print(f"  Generated: {worst['generated'][:150]}")

## 10. Save Results

Download these files from the Colab file browser (left panel).

In [ ]:
from datetime import datetime, timezone

# Save individual model results
for name, results, report_dict in [
    ("base_model", base_results, base_report_dict),
    ("umartransit", umar_results, umar_report_dict),
]:
    output = {
        "evaluated_at": datetime.now(timezone.utc).isoformat(),
        "report": report_dict,
        "results": results,
    }
    path = f"{name}_results.json"
    with open(path, "w") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"Saved: {path}")

# Save comparison
comparison = {
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
    "benchmark_version": benchmark["metadata"]["version"],
    "questions_evaluated": len(questions),
    "base_model": base_report_dict,
    "umartransit": umar_report_dict,
}
with open("comparison.json", "w") as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)
print("Saved: comparison.json")

print("\n" + "="*60)
print("DONE! Download these files from the left panel:")
print("  1. base_model_results.json")
print("  2. umartransit_results.json")
print("  3. comparison.json")
print("="*60)